# 01 - EDA Và Data Processing Cho Agent 1

Notebook này là bước xử lý dữ liệu đầu vào cho pipeline optimized.

Mục tiêu chính:
- Đọc duy nhất `data/dataset.csv`.
- Chuẩn hóa dữ liệu công nợ về schema AR chung.
- Tạo bảng invoice sạch, quality report, rolling snapshot features và current open invoice features.
- Bảo đảm feature lịch sử được tính theo nguyên tắc as-of cutoff T để tránh leakage.

Output của notebook này là input trực tiếp cho `02_baseline.ipynb`.

## 0. Cấu hình đường dẫn và schema

Khai báo đường dẫn input/output, mapping tên cột gốc sang schema AR và các constant dùng xuyên suốt notebook.

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path

pd.options.display.max_columns = 100

DATA_DIR = Path("../data")
PROCESSED_DIR = DATA_DIR / "processed"
RAW_DATA_PATH = DATA_DIR / "dataset.csv"
CLOSED_INVOICE_PATH = DATA_DIR / "dataset_closed_invoices.csv"
INVOICES_CLEAN_PATH = PROCESSED_DIR / "invoices_ar_clean.csv"
OPEN_INVOICES_PATH = PROCESSED_DIR / "open_invoices_current.csv"
INVOICE_CONFLICTS_PATH = PROCESSED_DIR / "invoice_id_conflicts.csv"
DATA_QUALITY_PATH = PROCESSED_DIR / "data_quality_report.csv"
SNAPSHOT_FEATURES_PATH = PROCESSED_DIR / "rolling_snapshot_features.csv"
SNAPSHOT_SUMMARY_PATH = PROCESSED_DIR / "rolling_snapshot_summary.csv"
CURRENT_OPEN_FEATURES_PATH = PROCESSED_DIR / "current_open_invoice_features.csv"

COLUMN_RENAME = {
    "cust_number": "customer_id",
    "posting_date": "issue_date",
    "due_in_date": "due_date",
    "clear_date": "paid_date",
    "total_open_amount": "amount_original",
    "invoice_currency": "currency",
    "cust_payment_terms": "payment_terms_code",
    "isOpen": "is_open",
}

FX_TO_USD = {
    "USD": 1.00,
    "CAD": 0.75,
}

HORIZONS = [30, 60, 90]
DATE_COLUMNS = ["issue_date", "due_date", "paid_date", "baseline_date"]
TEXT_COLUMNS = ["invoice_id", "customer_id", "name_customer", "business_code", "currency", "payment_terms_code"]
NUMERIC_COLUMNS = ["amount_original", "fx_to_usd", "amount"]
SORT_COLUMNS = ["issue_date", "invoice_id"]
COMMON_SCHEMA_COLUMNS = ["invoice_id", "customer_id", "issue_date", "due_date", "paid_date", "amount", "payment_terms", "is_open"]
QUALITY_FLAG_COLUMNS = [
    "payment_terms_source",
    "payment_terms_imputed",
    "payment_terms_invalid_after_fallback",
    "due_before_issue",
]
FULL_SCHEMA_COLUMNS = [
    "invoice_id",
    "customer_id",
    "name_customer",
    "business_code",
    "issue_date",
    "due_date",
    "paid_date",
    "baseline_date",
    "amount_original",
    "currency",
    "fx_to_usd",
    "amount",
    "payment_terms_code",
    "payment_terms",
    *QUALITY_FLAG_COLUMNS,
    "is_open",
]

## 1. Load dữ liệu gốc

Đọc file raw một lần từ `data/dataset.csv` và kiểm tra nhanh schema/dtype ban đầu.

In [ ]:
raw_df = pd.read_csv(RAW_DATA_PATH)
raw_df.info()

## 2. Hàm chuẩn hóa và kiểm soát chất lượng

Khối này gom các helper để parse ngày, chuẩn hóa tiền tệ, suy luận `payment_terms`, loại dòng trùng lặp, tạo quality flags và thêm outcome cho invoice đã đóng.

In [ ]:
# AGENT1_DATA_PROCESSING: utility chung cho chuẩn hóa AR dataset và feature as-of

def existing_columns(frame, columns):
    return [col for col in columns if col in frame.columns]


def order_columns(frame, priority_columns):
    priority = existing_columns(frame, priority_columns)
    rest = [col for col in frame.columns if col not in priority]
    return frame.loc[:, priority + rest]


def save_csv(frame, path):
    path.parent.mkdir(parents=True, exist_ok=True)
    frame.to_csv(path, index=False)
    return path


def parse_kaggle_date(series):
    values = series.astype("string").str.strip()
    parsed = pd.to_datetime(values, errors="coerce")
    yyyymmdd_mask = values.str.fullmatch(r"\d{8}", na=False)
    parsed.loc[yyyymmdd_mask] = pd.to_datetime(
        values.loc[yyyymmdd_mask], format="%Y%m%d", errors="coerce"
    )
    return parsed


def add_calendar_features(frame, date_columns_by_prefix, include_year=True):
    frame = frame.copy()
    parts = {"month": "month", "quarter": "quarter", "dayofweek": "dayofweek"}
    if include_year:
        parts = {"year": "year", **parts}

    for prefix, date_col in date_columns_by_prefix.items():
        dates = pd.to_datetime(frame[date_col], errors="coerce")
        for suffix, attr in parts.items():
            frame[f"{prefix}_{suffix}"] = getattr(dates.dt, attr).astype("Int64")
    return frame


def add_payment_outcomes(frame):
    frame = frame.copy()
    closed_mask = frame["is_open"].eq(0) & frame["paid_date"].notna()
    frame["invoice_life_days"] = (frame["paid_date"] - frame["issue_date"]).dt.days.astype("Int64")
    frame["days_late"] = (frame["paid_date"] - frame["due_date"]).dt.days.astype("Int64")
    frame.loc[~closed_mask, ["invoice_life_days", "days_late"]] = pd.NA
    frame["paid_on_or_before_due"] = frame["days_late"].le(0).fillna(False).astype("int8")
    return frame


def infer_payment_terms(frame):
    payment_days_from_baseline = (frame["due_date"] - frame["baseline_date"]).dt.days
    valid_code_days = payment_days_from_baseline.where(payment_days_from_baseline.gt(0))
    payment_terms_by_code = valid_code_days.groupby(frame["payment_terms_code"], dropna=False).median().round()

    term_from_code = pd.to_numeric(frame["payment_terms_code"].map(payment_terms_by_code), errors="coerce")
    fallback_terms = (frame["due_date"] - frame["issue_date"]).dt.days
    valid_fallback_terms = fallback_terms.where(fallback_terms.gt(0))
    global_valid_term = valid_fallback_terms.dropna().median()
    if pd.isna(global_valid_term):
        global_valid_term = valid_code_days.dropna().median()
    if pd.isna(global_valid_term):
        global_valid_term = 30

    payment_terms = term_from_code.copy()
    source = pd.Series("term_code_median", index=frame.index, dtype="string")

    invalid_from_code = payment_terms.isna() | payment_terms.le(0)
    payment_terms.loc[invalid_from_code] = valid_fallback_terms.loc[invalid_from_code]
    source.loc[invalid_from_code & valid_fallback_terms.notna()] = "due_minus_issue"

    still_invalid = payment_terms.isna() | payment_terms.le(0)
    payment_terms.loc[still_invalid] = global_valid_term
    source.loc[still_invalid] = "global_valid_median"

    invalid_after_fallback = payment_terms.isna() | payment_terms.le(0)
    return payment_terms.round().astype("Int64"), source, invalid_from_code, invalid_after_fallback


def standardize_raw_invoices(raw):
    frame = raw.rename(columns=COLUMN_RENAME).copy()
    quality_context = {
        "rows_raw": len(raw),
        "missing_invoice_id_raw": int(frame["invoice_id"].isna().sum()),
    }

    invoice_ids = pd.to_numeric(frame["invoice_id"], errors="coerce")
    quality_context["non_numeric_invoice_id_raw"] = int(frame["invoice_id"].notna().sum() - invoice_ids.notna().sum())
    frame = frame.loc[invoice_ids.notna()].copy()
    frame["invoice_id"] = invoice_ids.loc[frame.index].astype("int64").astype("string")
    quality_context["rows_after_valid_invoice_id"] = len(frame)

    for col in existing_columns(frame, TEXT_COLUMNS):
        frame[col] = frame[col].astype("string").str.strip()
    frame["currency"] = frame["currency"].str.upper()

    frame["issue_date"] = parse_kaggle_date(frame["issue_date"])
    frame["due_date"] = parse_kaggle_date(frame["due_date"])
    frame["paid_date"] = parse_kaggle_date(frame["paid_date"])
    frame["baseline_date"] = parse_kaggle_date(frame["baseline_create_date"])

    frame["amount_original"] = pd.to_numeric(frame["amount_original"], errors="coerce")
    frame["is_open"] = pd.to_numeric(frame["is_open"], errors="coerce").fillna(0).astype("int8")

    frame["fx_to_usd"] = frame["currency"].map(FX_TO_USD)
    unknown_currency = frame.loc[frame["fx_to_usd"].isna(), "currency"].dropna().unique()
    if len(unknown_currency):
        raise ValueError(f"Chưa có tỷ giá quy đổi USD cho currency: {unknown_currency.tolist()}")
    frame["amount"] = (frame["amount_original"] * frame["fx_to_usd"]).round(2)

    payment_terms, payment_terms_source, invalid_from_code, invalid_after_fallback = infer_payment_terms(frame)
    frame["payment_terms"] = payment_terms
    frame["payment_terms_source"] = payment_terms_source
    frame["payment_terms_imputed"] = payment_terms_source.ne("term_code_median").astype("int8")
    frame["payment_terms_invalid_after_fallback"] = invalid_after_fallback.fillna(True).astype("int8")
    frame["due_before_issue"] = frame["due_date"].lt(frame["issue_date"]).fillna(False).astype("int8")

    quality_context["payment_terms_invalid_from_code_rows"] = int(invalid_from_code.fillna(True).sum())
    quality_context["payment_terms_imputed_rows"] = int(frame["payment_terms_imputed"].sum())

    frame = frame.loc[:, existing_columns(frame, FULL_SCHEMA_COLUMNS)].copy()
    for col in DATE_COLUMNS:
        frame[col] = pd.to_datetime(frame[col], errors="coerce")
    for col in existing_columns(frame, NUMERIC_COLUMNS):
        frame[col] = pd.to_numeric(frame[col], errors="coerce")
    frame["payment_terms"] = pd.to_numeric(frame["payment_terms"], errors="coerce").round().astype("Int64")

    frame = add_payment_outcomes(frame)
    frame["amount_log1p"] = np.log1p(frame["amount"].clip(lower=0))
    frame = add_calendar_features(frame, {"issue": "issue_date", "due": "due_date", "paid": "paid_date"})

    dedupe_subset = existing_columns(frame, FULL_SCHEMA_COLUMNS)
    exact_duplicate_mask = frame.duplicated(subset=dedupe_subset, keep="first")
    quality_context["exact_duplicate_rows_collapsed"] = int(exact_duplicate_mask.sum())
    frame = frame.loc[~exact_duplicate_mask].copy()
    quality_context["rows_after_exact_dedup"] = len(frame)

    conflict_mask = frame["invoice_id"].duplicated(keep=False)
    invoice_id_conflicts = frame.loc[conflict_mask].sort_values(["invoice_id", "issue_date"]).copy()
    quality_context["invoice_id_conflict_rows_dropped"] = int(conflict_mask.sum())
    quality_context["invoice_id_conflict_groups"] = int(invoice_id_conflicts["invoice_id"].nunique())
    quality_context["invoice_id_conflicts"] = invoice_id_conflicts
    frame = frame.loc[~conflict_mask].copy()
    quality_context["rows_after_quality_gate"] = len(frame)

    return frame.sort_values(SORT_COLUMNS).reset_index(drop=True), quality_context


def build_data_quality_report(raw, invoices, quality_context):
    closed_mask = invoices["is_open"].eq(0)
    checks = [
        ("rows_raw", len(raw)),
        ("missing_invoice_id_raw", quality_context["missing_invoice_id_raw"]),
        ("non_numeric_invoice_id_raw", quality_context["non_numeric_invoice_id_raw"]),
        ("rows_after_valid_invoice_id", quality_context["rows_after_valid_invoice_id"]),
        ("exact_duplicate_rows_collapsed", quality_context["exact_duplicate_rows_collapsed"]),
        ("rows_after_exact_dedup", quality_context["rows_after_exact_dedup"]),
        ("invoice_id_conflict_groups", quality_context["invoice_id_conflict_groups"]),
        ("invoice_id_conflict_rows_dropped", quality_context["invoice_id_conflict_rows_dropped"]),
        ("rows_after_quality_gate", len(invoices)),
        ("duplicate_invoice_id_after_quality_gate", int(invoices["invoice_id"].duplicated().sum())),
        ("closed_invoices", int(closed_mask.sum())),
        ("open_invoices", int(invoices["is_open"].eq(1).sum())),
        ("closed_missing_paid_date", int((closed_mask & invoices["paid_date"].isna()).sum())),
        ("open_with_paid_date", int((invoices["is_open"].eq(1) & invoices["paid_date"].notna()).sum())),
        ("missing_issue_date", int(invoices["issue_date"].isna().sum())),
        ("missing_due_date", int(invoices["due_date"].isna().sum())),
        ("non_positive_amount_rows", int(invoices["amount"].le(0).sum())),
        ("payment_terms_invalid_from_code_rows", quality_context["payment_terms_invalid_from_code_rows"]),
        ("payment_terms_imputed_rows", int(invoices["payment_terms_imputed"].sum())),
        ("non_positive_payment_terms_after_fallback", int(invoices["payment_terms"].le(0).fillna(False).sum())),
        ("payment_terms_invalid_after_fallback", int(invoices["payment_terms_invalid_after_fallback"].sum())),
        ("due_before_issue_rows", int(invoices["due_before_issue"].sum())),
        ("closed_paid_before_issue", int((closed_mask & invoices["invoice_life_days"].lt(0)).sum())),
    ]
    return pd.DataFrame(checks, columns=["check", "value"])


def add_asof_core_features(frame, cutoff, include_target=False, mark_censored=False):
    frame = frame.copy()
    cutoff = pd.Timestamp(cutoff)
    for col in existing_columns(frame, ["issue_date", "due_date", "paid_date"]):
        frame[col] = pd.to_datetime(frame[col], errors="coerce")

    frame["cutoff_date"] = cutoff
    frame["cutoff_month"] = cutoff.month
    frame["cutoff_quarter"] = cutoff.quarter
    frame["cutoff_dayofweek"] = cutoff.dayofweek
    frame["observed_age_days"] = (cutoff - frame["issue_date"]).dt.days.clip(lower=0).astype("Int64")
    frame["days_until_due_at_T"] = (frame["due_date"] - cutoff).dt.days.astype("Int64")
    frame["days_past_due_at_T"] = (cutoff - frame["due_date"]).dt.days.clip(lower=0).astype("Int64")
    frame["is_overdue_at_T"] = frame["days_past_due_at_T"].fillna(0).gt(0).astype("int8")
    past_due = pd.to_numeric(frame["days_past_due_at_T"], errors="coerce").fillna(0)
    frame["aging_bucket_at_T"] = pd.cut(
        past_due,
        bins=[-1, 0, 30, 60, 90, np.inf],
        labels=["not_due", "1_30", "31_60", "61_90", "90_plus"],
    ).astype("string")
    frame["amount_log1p"] = np.log1p(pd.to_numeric(frame["amount"], errors="coerce").clip(lower=0))
    frame = add_calendar_features(frame, {"issue": "issue_date", "due": "due_date"}, include_year=False)

    if include_target:
        frame["remaining_days_at_T"] = (frame["paid_date"] - cutoff).dt.days.clip(lower=0).astype("Int64")
        frame["target_paid_date"] = frame["paid_date"]
        for horizon in HORIZONS:
            frame[f"paid_within_{horizon}d"] = frame["remaining_days_at_T"].le(horizon).fillna(False).astype("int8")
    if mark_censored:
        frame["is_censored_at_T"] = 1
    return frame


def eligible_invoice_pool(invoices, is_open):
    mask = (
        invoices["is_open"].eq(int(is_open))
        & invoices["issue_date"].notna()
        & invoices["due_date"].notna()
        & invoices["amount"].gt(0)
        & invoices["payment_terms"].notna()
        & invoices["payment_terms"].gt(0)
    )
    if "payment_terms_invalid_after_fallback" in invoices.columns:
        mask &= invoices["payment_terms_invalid_after_fallback"].eq(0)
    if not is_open:
        mask &= invoices["paid_date"].notna() & invoices["invoice_life_days"].ge(0).fillna(False)
        if "due_before_issue" in invoices.columns:
            mask &= invoices["due_before_issue"].eq(0)
    return invoices.loc[mask].copy()

## 3. Tạo clean AR table và quality report

Chuẩn hóa raw data thành `invoices_ar_clean`, tách current open invoices, ghi conflict report và quality report để baseline/model dùng cùng một nguồn dữ liệu sạch.

In [ ]:
# AGENT1_DATA_PROCESSING: tạo dataset sạch, open invoices hiện tại và quality report
invoices_ar_clean, data_quality_context = standardize_raw_invoices(raw_df)
invoice_id_conflicts = data_quality_context["invoice_id_conflicts"]
closed_invoices = order_columns(
    invoices_ar_clean.loc[invoices_ar_clean["is_open"].eq(0), COMMON_SCHEMA_COLUMNS].copy(),
    COMMON_SCHEMA_COLUMNS,
)

CURRENT_CUTOFF = invoices_ar_clean["issue_date"].max()
open_invoices_current = add_asof_core_features(
    eligible_invoice_pool(invoices_ar_clean, is_open=True),
    CURRENT_CUTOFF,
    mark_censored=True,
).sort_values(SORT_COLUMNS).reset_index(drop=True)

data_quality_report = build_data_quality_report(raw_df, invoices_ar_clean, data_quality_context)

save_csv(closed_invoices, CLOSED_INVOICE_PATH)
save_csv(invoices_ar_clean, INVOICES_CLEAN_PATH)
save_csv(open_invoices_current, OPEN_INVOICES_PATH)
save_csv(invoice_id_conflicts, INVOICE_CONFLICTS_PATH)
save_csv(data_quality_report, DATA_QUALITY_PATH)

print(f"Saved {len(closed_invoices):,} closed invoices to {CLOSED_INVOICE_PATH}")
print(f"Saved full clean AR table: {INVOICES_CLEAN_PATH} ({len(invoices_ar_clean):,} rows)")
print(f"Saved current open invoice table: {OPEN_INVOICES_PATH} ({len(open_invoices_current):,} rows)")
print(f"Saved invoice_id conflict report: {INVOICE_CONFLICTS_PATH} ({len(invoice_id_conflicts):,} rows)")
print(f"Current as-of cutoff inferred from latest issue_date: {CURRENT_CUTOFF.date()}")
data_quality_report

## 4. Helper feature as-of cutoff T

Các feature lịch sử như customer median delay, recency delay, term median và global median chỉ được tính từ invoice đã trả trước hoặc đúng cutoff T. Đây là lớp chống leakage quan trọng nhất của notebook.

In [ ]:
# AGENT1_DATA_PROCESSING: helper tạo feature lịch sử as-of T, chỉ dùng dữ liệu trước cutoff
CUSTOMER_HISTORY_COLUMNS = [
    "customer_id",
    "customer_invoice_count_hist",
    "customer_median_delay_hist",
    "customer_mean_delay_hist",
    "customer_median_life_hist",
    "customer_mean_life_hist",
    "customer_median_amount_hist",
    "customer_mean_amount_hist",
    "customer_paid_amount_hist",
    "customer_last_paid_date",
    "customer_recency_delay_median_hist",
    "customer_recency_life_median_hist",
]
TERM_HISTORY_COLUMNS = [
    "payment_terms",
    "term_invoice_count_hist",
    "term_median_delay_hist",
    "term_mean_delay_hist",
    "term_median_life_hist",
]
GLOBAL_HISTORY_DEFAULTS = {
    "global_invoice_count_hist": 0,
    "global_median_delay_hist": 0.0,
    "global_mean_delay_hist": 0.0,
    "global_median_life_hist": 0.0,
    "global_mean_life_hist": 0.0,
    "global_median_amount_hist": 0.0,
    "global_mean_amount_hist": 0.0,
}
NUMERIC_FALLBACKS = {
    "term_median_delay_hist": ["global_median_delay_hist"],
    "term_mean_delay_hist": ["global_mean_delay_hist"],
    "term_median_life_hist": ["global_median_life_hist"],
    "customer_median_delay_hist": ["term_median_delay_hist", "global_median_delay_hist"],
    "customer_mean_delay_hist": ["term_mean_delay_hist", "global_mean_delay_hist"],
    "customer_median_life_hist": ["term_median_life_hist", "global_median_life_hist"],
    "customer_mean_life_hist": ["global_mean_life_hist"],
    "customer_median_amount_hist": ["global_median_amount_hist"],
    "customer_mean_amount_hist": ["global_mean_amount_hist"],
    "customer_recency_delay_median_hist": ["customer_median_delay_hist"],
    "customer_recency_life_median_hist": ["customer_median_life_hist"],
}


def prepare_paid_history(history):
    history = history.dropna(subset=["customer_id", "issue_date", "due_date", "paid_date"]).copy()
    history = history.loc[pd.to_numeric(history["amount"], errors="coerce").gt(0)].copy()
    history["delay_days"] = (history["paid_date"] - history["due_date"]).dt.days
    history["invoice_life_days"] = (history["paid_date"] - history["issue_date"]).dt.days
    return history.loc[history["invoice_life_days"].ge(0)].copy()


def build_history_features(history):
    history = prepare_paid_history(history)
    if history.empty:
        return (
            pd.DataFrame(columns=CUSTOMER_HISTORY_COLUMNS),
            pd.DataFrame(columns=TERM_HISTORY_COLUMNS),
            GLOBAL_HISTORY_DEFAULTS.copy(),
            np.array([], dtype=float),
        )

    amount_history = np.sort(pd.to_numeric(history["amount"], errors="coerce").dropna().to_numpy(dtype=float))
    global_stats = {
        "global_invoice_count_hist": int(len(history)),
        "global_median_delay_hist": float(history["delay_days"].median()),
        "global_mean_delay_hist": float(history["delay_days"].mean()),
        "global_median_life_hist": float(history["invoice_life_days"].median()),
        "global_mean_life_hist": float(history["invoice_life_days"].mean()),
        "global_median_amount_hist": float(history["amount"].median()),
        "global_mean_amount_hist": float(history["amount"].mean()),
    }

    customer_hist = (
        history.groupby("customer_id", dropna=False)
        .agg(
            customer_invoice_count_hist=("invoice_id", "count"),
            customer_median_delay_hist=("delay_days", "median"),
            customer_mean_delay_hist=("delay_days", "mean"),
            customer_median_life_hist=("invoice_life_days", "median"),
            customer_mean_life_hist=("invoice_life_days", "mean"),
            customer_median_amount_hist=("amount", "median"),
            customer_mean_amount_hist=("amount", "mean"),
            customer_paid_amount_hist=("amount", "sum"),
            customer_last_paid_date=("paid_date", "max"),
        )
        .reset_index()
    )

    recent_hist = (
        history.sort_values("paid_date")
        .groupby("customer_id", dropna=False)
        .tail(5)
        .groupby("customer_id", dropna=False)
        .agg(
            customer_recency_delay_median_hist=("delay_days", "median"),
            customer_recency_life_median_hist=("invoice_life_days", "median"),
        )
        .reset_index()
    )
    customer_hist = customer_hist.merge(recent_hist, on="customer_id", how="left")

    term_hist = (
        history.groupby("payment_terms", dropna=False)
        .agg(
            term_invoice_count_hist=("invoice_id", "count"),
            term_median_delay_hist=("delay_days", "median"),
            term_mean_delay_hist=("delay_days", "mean"),
            term_median_life_hist=("invoice_life_days", "median"),
        )
        .reset_index()
    )
    return customer_hist, term_hist, global_stats, amount_history


def fill_numeric_with_backoff(frame, column, fallbacks):
    values = pd.to_numeric(frame[column], errors="coerce") if column in frame else pd.Series(np.nan, index=frame.index)
    for fallback in fallbacks:
        if isinstance(fallback, str) and fallback in frame:
            values = values.fillna(pd.to_numeric(frame[fallback], errors="coerce"))
        else:
            values = values.fillna(fallback)
    return values


def attach_history_features(frame, history, cutoff):
    customer_hist, term_hist, global_stats, amount_history = build_history_features(history)
    frame = frame.merge(customer_hist, on="customer_id", how="left").merge(term_hist, on="payment_terms", how="left")

    for key, value in global_stats.items():
        frame[key] = value
    for col in ["customer_invoice_count_hist", "term_invoice_count_hist"]:
        frame[col] = pd.to_numeric(frame[col], errors="coerce").fillna(0).astype("int64")
    for col, fallback_keys in NUMERIC_FALLBACKS.items():
        fallbacks = [global_stats[key] if key in global_stats else key for key in fallback_keys]
        frame[col] = fill_numeric_with_backoff(frame, col, fallbacks)

    frame["customer_has_history"] = frame["customer_invoice_count_hist"].gt(0).astype("int8")
    frame["customer_history_sufficient"] = frame["customer_invoice_count_hist"].ge(3).astype("int8")
    frame["term_has_history"] = frame["term_invoice_count_hist"].gt(0).astype("int8")
    frame["customer_paid_amount_hist"] = pd.to_numeric(frame["customer_paid_amount_hist"], errors="coerce").fillna(0)
    frame["customer_last_paid_date"] = pd.to_datetime(frame["customer_last_paid_date"], errors="coerce")
    frame["customer_last_paid_age_days"] = (pd.Timestamp(cutoff) - frame["customer_last_paid_date"]).dt.days.astype("Int64")

    amount_values = pd.to_numeric(frame["amount"], errors="coerce").fillna(0).to_numpy(dtype=float)
    if len(amount_history):
        frame["amount_percentile_hist"] = np.searchsorted(amount_history, amount_values, side="right") / len(amount_history)
    else:
        frame["amount_percentile_hist"] = 0.5

    customer_median_amount = pd.to_numeric(frame["customer_median_amount_hist"], errors="coerce")
    denominator = customer_median_amount.where(customer_median_amount.gt(0), pd.NA)
    frame["amount_vs_customer_median_hist"] = (pd.to_numeric(frame["amount"], errors="coerce") / denominator).fillna(1.0)
    return frame


def add_asof_data_features(open_frame, history, cutoff, include_target=True):
    frame = add_asof_core_features(
        open_frame,
        cutoff,
        include_target=include_target,
        mark_censored=not include_target,
    )
    return attach_history_features(frame, history, cutoff)

## 5. Rolling snapshot/backtest dataset

Tạo các cutoff theo thời gian để mô phỏng production: tại ngày T, invoice nào đang mở, còn bao nhiêu ngày mới trả, có quá hạn hay không và có trả trong 30/60/90 ngày không.

In [ ]:
# AGENT1_DATA_PROCESSING: tạo rolling snapshot/backtest dataset từ invoice đã đóng
WARMUP_DAYS = 90
MIN_HISTORY_ROWS = 250
CUTOFF_FREQ = "W-FRI"
SNAPSHOT_PRIORITY_COLUMNS = [
    "cutoff_date",
    "time_split",
    "invoice_id",
    "customer_id",
    "issue_date",
    "due_date",
    "paid_date",
    "target_paid_date",
    "amount",
    "amount_log1p",
    "amount_percentile_hist",
    "amount_vs_customer_median_hist",
    "payment_terms",
    "observed_age_days",
    "days_until_due_at_T",
    "days_past_due_at_T",
    "is_overdue_at_T",
    "aging_bucket_at_T",
    "cutoff_month",
    "cutoff_quarter",
    "cutoff_dayofweek",
    "remaining_days_at_T",
    "paid_within_30d",
    "paid_within_60d",
    "paid_within_90d",
    "customer_invoice_count_hist",
    "customer_has_history",
    "customer_history_sufficient",
    "customer_median_delay_hist",
    "customer_recency_delay_median_hist",
    "customer_median_life_hist",
    "customer_median_amount_hist",
    "customer_last_paid_age_days",
    "term_invoice_count_hist",
    "term_has_history",
    "term_median_delay_hist",
    "global_invoice_count_hist",
    "global_median_delay_hist",
]


def assign_time_split(frame, train_frac=0.60, validation_frac=0.20):
    frame = frame.copy()
    cutoffs = pd.Series(pd.to_datetime(frame["cutoff_date"].unique())).sort_values().reset_index(drop=True)
    n_cutoffs = len(cutoffs)
    if n_cutoffs < 3:
        raise ValueError("Cần ít nhất 3 cutoff để tạo train/validation/test theo thời gian.")

    train_n = max(1, int(np.floor(n_cutoffs * train_frac)))
    validation_n = max(1, int(np.floor(n_cutoffs * validation_frac)))
    if train_n + validation_n >= n_cutoffs:
        train_n = max(1, n_cutoffs - 2)
        validation_n = 1

    split_map = {}
    for i, cutoff in enumerate(cutoffs):
        if i < train_n:
            split = "train"
        elif i < train_n + validation_n:
            split = "validation"
        else:
            split = "test"
        split_map[pd.Timestamp(cutoff)] = split

    frame["time_split"] = pd.to_datetime(frame["cutoff_date"]).map(split_map).astype("string")
    return frame


closed_invoice_base = eligible_invoice_pool(invoices_ar_clean, is_open=False)
cutoff_start = closed_invoice_base["issue_date"].min() + pd.Timedelta(days=WARMUP_DAYS)
cutoff_end = closed_invoice_base["paid_date"].max() - pd.Timedelta(days=max(HORIZONS))
if cutoff_end <= cutoff_start:
    cutoff_end = closed_invoice_base["paid_date"].max() - pd.Timedelta(days=30)

snapshot_frames = []
for cutoff in pd.date_range(cutoff_start, cutoff_end, freq=CUTOFF_FREQ):
    history = closed_invoice_base.loc[closed_invoice_base["paid_date"].le(cutoff)]
    open_at_t = closed_invoice_base.loc[
        closed_invoice_base["issue_date"].le(cutoff) & closed_invoice_base["paid_date"].gt(cutoff)
    ]
    if len(history) >= MIN_HISTORY_ROWS and not open_at_t.empty:
        snapshot_frames.append(add_asof_data_features(open_at_t, history, cutoff, include_target=True))

if not snapshot_frames:
    raise ValueError("Không tạo được rolling snapshot nào; kiểm tra date range hoặc MIN_HISTORY_ROWS.")

rolling_snapshot_features = pd.concat(snapshot_frames, ignore_index=True)
rolling_snapshot_features = assign_time_split(rolling_snapshot_features)
rolling_snapshot_features = order_columns(rolling_snapshot_features, SNAPSHOT_PRIORITY_COLUMNS)
rolling_snapshot_summary = (
    rolling_snapshot_features.groupby(["time_split", "cutoff_date"])
    .agg(
        open_invoice_rows=("invoice_id", "count"),
        open_amount=("amount", "sum"),
        overdue_invoice_rows=("is_overdue_at_T", "sum"),
        median_observed_age_days=("observed_age_days", "median"),
        median_remaining_days=("remaining_days_at_T", "median"),
    )
    .reset_index()
    .sort_values("cutoff_date")
    .reset_index(drop=True)
)

save_csv(rolling_snapshot_features, SNAPSHOT_FEATURES_PATH)
save_csv(rolling_snapshot_summary, SNAPSHOT_SUMMARY_PATH)

split_cutoffs = rolling_snapshot_features.groupby("time_split")["cutoff_date"].nunique().to_dict()
print(f"Saved rolling snapshot features: {SNAPSHOT_FEATURES_PATH}")
print(f"Saved rolling snapshot summary: {SNAPSHOT_SUMMARY_PATH}")
print(
    f"Snapshots: {rolling_snapshot_features['cutoff_date'].nunique():,} cutoffs, "
    f"{len(rolling_snapshot_features):,} invoice-as-of rows"
)
print(
    f"Cutoff range: {rolling_snapshot_features['cutoff_date'].min().date()} "
    f"to {rolling_snapshot_features['cutoff_date'].max().date()}"
)
print(f"Cutoffs by time split: {split_cutoffs}")
rolling_snapshot_features.head()

## 6. Current open invoice features

Tạo feature cho các invoice đang mở ở thời điểm hiện tại. Artifact này dùng cho bước forecast/inference sau khi baseline hoặc model đã được chọn.

In [ ]:
# AGENT1_DATA_PROCESSING: tạo feature as-of cho invoice đang mở, chỉ chuẩn bị dữ liệu
CURRENT_OPEN_PRIORITY_COLUMNS = [
    "cutoff_date",
    "invoice_id",
    "customer_id",
    "name_customer",
    "issue_date",
    "due_date",
    "amount",
    "amount_log1p",
    "amount_percentile_hist",
    "amount_vs_customer_median_hist",
    "payment_terms",
    "observed_age_days",
    "days_until_due_at_T",
    "days_past_due_at_T",
    "is_overdue_at_T",
    "aging_bucket_at_T",
    "cutoff_month",
    "cutoff_quarter",
    "cutoff_dayofweek",
    "is_censored_at_T",
    "customer_invoice_count_hist",
    "customer_has_history",
    "customer_history_sufficient",
    "customer_median_delay_hist",
    "customer_recency_delay_median_hist",
    "customer_median_life_hist",
    "customer_median_amount_hist",
    "customer_last_paid_age_days",
    "term_invoice_count_hist",
    "term_has_history",
    "term_median_delay_hist",
    "global_invoice_count_hist",
    "global_median_delay_hist",
]

history_for_current = closed_invoice_base.loc[closed_invoice_base["paid_date"].le(CURRENT_CUTOFF)]
current_open_invoice_features = add_asof_data_features(
    eligible_invoice_pool(invoices_ar_clean, is_open=True),
    history_for_current,
    CURRENT_CUTOFF,
    include_target=False,
)
current_open_invoice_features = order_columns(current_open_invoice_features, CURRENT_OPEN_PRIORITY_COLUMNS)
current_open_invoice_features = current_open_invoice_features.sort_values(SORT_COLUMNS).reset_index(drop=True)

save_csv(current_open_invoice_features, CURRENT_OPEN_FEATURES_PATH)

print(f"Saved current open invoice features: {CURRENT_OPEN_FEATURES_PATH} ({len(current_open_invoice_features):,} rows)")
current_open_invoice_features.head()

## 7. Kiểm tra cuối và artifact summary

Chạy các assertion tối thiểu để bảo đảm dữ liệu sạch không còn duplicate invoice id, không còn `payment_terms` không hợp lệ, rolling snapshot có đủ target và các file output đã được ghi ra `data/processed`.

In [ ]:
# AGENT1_DATA_PROCESSING: kiểm tra cuối cho optimized data pipeline
assert not invoices_ar_clean["invoice_id"].duplicated().any(), "Clean AR table vẫn còn duplicate invoice_id."
assert invoices_ar_clean["payment_terms"].dropna().gt(0).all(), "payment_terms <= 0 vẫn lọt vào clean table."
assert rolling_snapshot_features["time_split"].notna().all(), "Rolling snapshot chưa có split theo cutoff."
assert rolling_snapshot_features["cutoff_date"].nunique() >= 3, "Không đủ cutoff cho train/validation/test."
assert current_open_invoice_features["is_censored_at_T"].eq(1).all(), "Open invoice hiện tại phải được đánh dấu censored."

processing_summary = pd.DataFrame(
    [
        ("clean_invoices", len(invoices_ar_clean)),
        ("closed_invoices_for_backtest", len(closed_invoice_base)),
        ("current_open_invoices_for_inference", len(current_open_invoice_features)),
        ("rolling_snapshot_rows", len(rolling_snapshot_features)),
        ("rolling_snapshot_cutoffs", rolling_snapshot_features["cutoff_date"].nunique()),
        ("train_cutoffs", rolling_snapshot_features.loc[rolling_snapshot_features["time_split"].eq("train"), "cutoff_date"].nunique()),
        ("validation_cutoffs", rolling_snapshot_features.loc[rolling_snapshot_features["time_split"].eq("validation"), "cutoff_date"].nunique()),
        ("test_cutoffs", rolling_snapshot_features.loc[rolling_snapshot_features["time_split"].eq("test"), "cutoff_date"].nunique()),
    ],
    columns=["artifact", "value"],
)
processing_summary